# Comparison of deconvolver calibrators

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from EDA.edautils import plot_deconvolution_results
from methyldl.deconvolution.evaluation import compute_deconvolution_metrics
from methyldl.deconvolution.linear_calibrator import LinearCalibrator

%load_ext autoreload
%autoreload 2

## Utility functions

In [2]:
def plot_mixtures_pred_vs_true(
    ground_truth_mixture: np.ndarray,
    predicted_mixtures: list[np.ndarray],
    predicted_mixture_labels: list[str],
    width: float = 0.2,
    title: str = "Predicted Mixtures vs Ground Truth Mixture",
    ctype_names: list[str] = None,
):
    """
    Plot the predicted mixtures against the ground truth mixture for a single sample.
    The function creates a bar plot where the x axis represents the cell types and the y axis represents the mixture proportions.
    Each predicted mixture is plotted as a separate bar. The ground truth mixture is plotted as red dots for each cell type.

    Args:
        ground_truth_mixture: A 1D array of shape (n_cell_types,) representing the true mixture proportions.
        predicted_mixtures: A list of 1D arrays, each of shape (n_cell_types,), representing the predicted mixture proportions from different models.
        predicted_mixture_labels: A list of strings representing the labels for each predicted mixture (e.g., model names).
    """
    assert len(predicted_mixtures) == len(
        predicted_mixture_labels
    ), "Number of predicted mixtures must match number of labels"
    assert all(
        pred.shape == ground_truth_mixture.shape for pred in predicted_mixtures
    ), "All predicted mixtures must have the same shape as the ground truth mixture"
    plt.figure(figsize=(8, 6))
    n_cell_types = len(ground_truth_mixture)
    n_predicted_mixtures = len(predicted_mixtures)
    x = np.arange(n_cell_types)  # start of the the label locations
    x_center = (
        x + width * (n_predicted_mixtures - 1) / 2
    )  # middle of the group of bars for each cell type

    # Plot the ground truth mixture as red dots
    plt.scatter(
        x_center, ground_truth_mixture, color="red", zorder=-1, label="Ground Truth"
    )

    # Plot each predicted mixture as a bar
    for i, (predicted_mixture, label) in enumerate(
        zip(predicted_mixtures, predicted_mixture_labels)
    ):
        plt.bar(x + i * width, predicted_mixture, width, label=label)

    # plot the cell types names on the x axis, rotated by 90 degrees
    if ctype_names is not None:
        plt.xticks(x_center, ctype_names, rotation=90)
    plt.ylabel("Mixture Proportions")
    plt.title(title)
    plt.legend()
    plt.grid()
    plt.tight_layout()
    plt.show()

## Data loading

In [3]:
with open("../App/labels_dict.json", "r") as f:
    labels_to_ctype_names = json.load(f)
with open("../App/old_labels_dict.json", "r") as f:
    old_labels_to_ctype_names = json.load(f)
old_labels_to_ctype_names = {int(k): v for k, v in old_labels_to_ctype_names.items()}
labels_to_ctype_names = {int(k): v for k, v in labels_to_ctype_names.items()}
ctype_names_to_labels = {v: k for k, v in labels_to_ctype_names.items()}
ctype_names_list = list(ctype_names_to_labels.keys())

In [4]:
# compute the mapping from old labels to new labels (permutation of the labels)
old_to_new_label_mapping = {old_label: ctype_names_to_labels[old_name] for old_label, old_name in old_labels_to_ctype_names.items()}

In [5]:
target_proportions_file = Path("../Data/training_data/mixture_predictions/uxm/target_proportions.npz")
target_prop = np.load(target_proportions_file)["arr_0"]

In [6]:
uxm_pred_folder = Path("../Data/training_data/mixture_predictions/uxm")
uxm_train_file = uxm_pred_folder / "uxm_results_train.npz"
uxm_val_file = uxm_pred_folder / "uxm_results_valid.npz"
uxm_test_file = uxm_pred_folder / "uxm_results_test.npz"

uxm_train_data = np.load(uxm_train_file)["arr_0"]
uxm_val_data = np.load(uxm_val_file)["arr_0"]
uxm_test_data = np.load(uxm_test_file)["arr_0"]

## Compare calibrators

In [31]:
def wrapper_comp_metrics(test_pred: dict, round_: int=6) -> pd.DataFrame:
    results = {
        method_name: compute_deconvolution_metrics(
            pred=pred,
            target=target_prop,
            class_names=ctype_names_list
        )
        for method_name, pred in test_pred.items()
    }
    results = {
        k: {metric: value for metric, value in v.items() if "per_class" not in metric}
        for k, v in results.items()
    }
    results_df = pd.DataFrame(results).T
    float_cols = ["mae", "mse", "kl", 'max_error', 'cosine_sim', "loa_lower", "loa_upper", "loa_width", "worst_class_loa_lower", "worst_class_loa_upper", "worst_class_loa_width"]
    for col in float_cols:
        results_df[col] = results_df[col].astype(float).round(round_)
    results_df.sort_values("mse", inplace=True)
    return results_df, results

In [8]:
linear_cal = LinearCalibrator()
linear_cal.fit(uxm_val_data, target_prop)
uxm_test_pred_lin_cal_clip_norm, raw_uxm_test_pred_lin_cal  = linear_cal.predict(uxm_test_data, norm_method="clip-normalize")
uxm_test_pred_lin_cal_clip_norm_no_upper_clip,_  = linear_cal.predict(uxm_test_data, upper_clip=False, norm_method="clip-normalize")
uxm_test_pred_lin_cal_softmax,_  = linear_cal.predict(uxm_test_data, norm_method="softmax")
uxm_test_pred_lin_cal_simplex_proj, _ = linear_cal.predict(uxm_test_data, norm_method="simplex-projection")
uxm_test_pred_lin_cal_shift_norm, _ = linear_cal.predict(uxm_test_data, norm_method="shift-normalize")
uxm_test_pred_lin_cal_entmax, _ = linear_cal.predict(uxm_test_data, norm_method="entmax", entmax_alpha=1.5)

In [32]:
results_df, results = wrapper_comp_metrics(
    {
        "UXM + no cal": uxm_test_data,
        "UXM + Lin cal (clip-norm)": uxm_test_pred_lin_cal_clip_norm,
        "UXM + Lin cal (clip-norm no upper clip)": uxm_test_pred_lin_cal_clip_norm_no_upper_clip,
        "UXM + Lin cal (softmax)": uxm_test_pred_lin_cal_softmax,
        "UXM + Lin cal (simplex-projection)": uxm_test_pred_lin_cal_simplex_proj,
        "UXM + Lin cal (shift-normalize)": uxm_test_pred_lin_cal_shift_norm,
        "UXM + Lin cal (entmax)": uxm_test_pred_lin_cal_entmax
    }
)
results_df

,mae,mse,kl,max_error,cosine_sim,loa_lower,loa_upper,loa_width,worst_class_idx,worst_class_name,worst_class_loa_lower,worst_class_loa_upper,worst_class_loa_width
UXM + Lin cal (simplex-projection),0.004211,0.000208,0.124543,0.363513,0.986401,-0.028240,0.028240,0.056480,11,Colon-Fibro,-0.063390,0.085064,0.148454
UXM + Lin cal (clip-norm no upper clip),0.005028,0.000254,0.133447,0.411706,0.985780,-0.031260,0.031260,0.062520,11,Colon-Fibro,-0.063620,0.085536,0.149156
UXM + Lin cal (clip-norm),0.005033,0.000255,0.133534,0.411706,0.985778,-0.031280,0.031280,0.062561,11,Colon-Fibro,-0.063627,0.085552,0.149179
UXM + no cal,0.005563,0.000294,0.145682,0.513800,0.985040,-0.033601,0.033601,0.067202,11,Colon-Fibro,-0.068714,0.095688,0.164402
UXM + Lin cal (shift-normalize),0.009969,0.000536,0.227500,0.481808,0.983179,-0.045368,0.045368,0.090736,11,Colon-Fibro,-0.068851,0.083535,0.152386
UXM + Lin cal (entmax),0.034214,0.004380,1.125068,0.747294,0.798118,-0.129715,0.129715,0.259429,13,Endothel,-0.141263,0.138283,0.279546
UXM + Lin cal (softmax),0.042996,0.007732,1.994659,0.948445,0.416696,-0.172344,0.172344,0.344688,32,Pancreas-Delta,-0.177546,0.175832,0.353379


In [ ]:
sample_idx = 1
plot_mixtures_pred_vs_true(
    ground_truth_mixture=target_prop[sample_idx],
    predicted_mixtures=[
        uxm_test_data[sample_idx],
        uxm_test_pred_lin_cal_clip_norm[sample_idx],
        uxm_test_pred_lin_cal_clip_norm_no_upper_clip[sample_idx],
        uxm_test_pred_lin_cal_softmax[sample_idx],
    ],
    predicted_mixture_labels=[
        "UXM + no cal",
        "UXM + Lin cal (clip-norm)",
        "UXM + Lin cal (no upper clip)",
        "UXM + Lin cal (softmax)"
    ],
    title="UXM Predictions with and without Linear Calibration",
    ctype_names=ctype_names_list
)

Next step is to try :

- vector scaling on the log prob (what the Dir Cal paper does)
- vector scaling on the prob
- matrix scaling on the log prob (= Dir Cal)
- matrix scaling on the prob (= MS)

with alpha entmax (alpha = 1, 1.5, 2)